In [ ]:
# ============================================================
# 03 Color Feature Engineering Model
# Path configuration and output directories
# ============================================================

from pathlib import Path

CURRENT_DIR = Path.cwd()

# 从当前目录开始，逐级向上查找项目根目录
# 兼容 notebook 和 notebooks 两种文件夹命名
PROJECT_DIR = None

for path in [CURRENT_DIR] + list(CURRENT_DIR.parents):
    has_data = (path / "data").exists()
    has_notebook_dir = (path / "notebook").exists() or (path / "notebooks").exists()
    
    if has_data and has_notebook_dir:
        PROJECT_DIR = path
        break

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Cannot locate project root. Please check whether the notebook is inside the kaggle-star-type-prediction project."
    )

# 数据目录
DATA_DIR = PROJECT_DIR / "data" / "raw"

# 颜色特征工程结果目录
REPORT_DIR = PROJECT_DIR / "reports" / "model" / "color_features"

# 两个模型的输出目录
RF_COLOR_DIR = REPORT_DIR / "random_forest"
ET_COLOR_DIR = REPORT_DIR / "extra_trees"

# 自动创建tables和figures文件夹
for model_dir in [RF_COLOR_DIR, ET_COLOR_DIR]:
    (model_dir / "tables").mkdir(parents=True, exist_ok=True)
    (model_dir / "figures").mkdir(parents=True, exist_ok=True)

print("Current directory:", CURRENT_DIR)
print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)
print("Report directory:", REPORT_DIR)
print("Output directories are ready.")

In [ ]:
# ============================================================
# 读取数据
# ============================================================

import pandas as pd
from sklearn.model_selection import train_test_split

# 读取训练数据
train_path = DATA_DIR / "train.csv"
train_df = pd.read_csv(train_path)

print("Train shape before feature engineering:", train_df.shape)
print("Columns:")
print(train_df.columns.tolist())

In [ ]:
# ============================================================
# Color index feature engineering
# ============================================================

# 检查颜色指数所需的原始光度特征是否存在
magnitude_cols = ["u", "g", "r", "i", "z"]

missing_cols = [col for col in magnitude_cols if col not in train_df.columns]
if missing_cols:
    raise ValueError(f"Missing magnitude columns: {missing_cols}")

# 复制一份数据，避免直接改动原始DataFrame
train_color_df = train_df.copy()

# 构造颜色指数特征
train_color_df["u_g"] = train_color_df["u"] - train_color_df["g"]
train_color_df["g_r"] = train_color_df["g"] - train_color_df["r"]
train_color_df["r_i"] = train_color_df["r"] - train_color_df["i"]
train_color_df["i_z"] = train_color_df["i"] - train_color_df["z"]

color_feature_cols = ["u_g", "g_r", "r_i", "i_z"]

print("Added color index features:")
print(color_feature_cols)

print("Train shape after feature engineering:", train_color_df.shape)

train_color_df[color_feature_cols].head()

In [ ]:
# ============================================================
# 准备训练集和验证集
# ============================================================

# 自动识别目标列
possible_target_cols = ["class", "target", "label", "Class"]

target_col = None
for col in possible_target_cols:
    if col in train_color_df.columns:
        target_col = col
        break

if target_col is None:
    raise ValueError(
        "Cannot find target column. Please check whether the label column is named class, target, label, or Class."
    )

print("Target column:", target_col)

# 如果存在id列，不作为模型特征
drop_cols = [target_col]
if "id" in train_color_df.columns:
    drop_cols.append("id")

X_color = train_color_df.drop(columns=drop_cols)
y = train_color_df[target_col]

# 与baseline阶段保持一致的划分方式
X_train_color, X_valid_color, y_train, y_valid = train_test_split(
    X_color,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train_color shape:", X_train_color.shape)
print("X_valid_color shape:", X_valid_color.shape)
print("y_train distribution:")
print(y_train.value_counts(normalize=True))

print("y_valid distribution:")
print(y_valid.value_counts(normalize=True))

In [ ]:
# ============================================================
# 预处理、评估、保存分类报告、保存混淆矩阵、保存特征重要性
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# ------------------------------------------------------------
# Build preprocessing pipeline
# ------------------------------------------------------------

def build_preprocessor(X):
    """
    Build preprocessing pipeline for numeric and categorical features.
    Numeric features: median imputation.
    Categorical features: most frequent imputation + one-hot encoding.
    """

    numeric_cols = X.select_dtypes(
        include=["number"]
    ).columns.tolist()

    categorical_cols = X.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    try:
        onehot = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    except TypeError:
        onehot = OneHotEncoder(
            handle_unknown="ignore",
            sparse=False
        )

    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", onehot)
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols)
        ],
        remainder="drop"
    )

    return preprocessor, numeric_cols, categorical_cols


# ------------------------------------------------------------
# Extract feature names after preprocessing
# ------------------------------------------------------------

def get_preprocessed_feature_names(pipeline):
    """
    Get feature names after ColumnTransformer preprocessing.
    """

    preprocessor = pipeline.named_steps["preprocess"]
    feature_names = preprocessor.get_feature_names_out()

    cleaned_feature_names = []

    for name in feature_names:
        name = name.replace("num__", "")
        name = name.replace("cat__", "")
        name = name.replace("remainder__", "")
        cleaned_feature_names.append(name)

    return cleaned_feature_names


# ------------------------------------------------------------
# Evaluate model and save outputs
# ------------------------------------------------------------

def evaluate_and_save_model(
    model_name,
    pipeline,
    X_valid,
    y_valid,
    output_dir,
    labels=None
):
    """
    Evaluate a fitted model pipeline and save:
    1. classification report
    2. confusion matrix figure
    3. feature importance table
    4. top 20 feature importance figure
    """

    tables_dir = output_dir / "tables"
    figures_dir = output_dir / "figures"

    tables_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)

    if labels is None:
        labels = sorted(y_valid.unique())

    # Predict
    y_pred = pipeline.predict(X_valid)

    # Metrics
    accuracy = accuracy_score(y_valid, y_pred)
    balanced_acc = balanced_accuracy_score(y_valid, y_pred)
    macro_f1 = f1_score(y_valid, y_pred, average="macro")

    print(f"{model_name} results")
    print("=" * 60)
    print(f"Accuracy: {accuracy:.6f}")
    print(f"Balanced Accuracy: {balanced_acc:.6f}")
    print(f"Macro F1: {macro_f1:.6f}")
    print()
    print(classification_report(y_valid, y_pred, digits=6, zero_division=0))

    # Save text report
    report_text = []
    report_text.append(f"{model_name} Color Feature Report")
    report_text.append("=" * 60)
    report_text.append(f"Accuracy: {accuracy:.6f}")
    report_text.append(f"Balanced Accuracy: {balanced_acc:.6f}")
    report_text.append(f"Macro F1: {macro_f1:.6f}")
    report_text.append("")
    report_text.append("Classification Report:")
    report_text.append(
        classification_report(
            y_valid,
            y_pred,
            digits=6,
            zero_division=0
        )
    )

    report_path = tables_dir / f"{model_name.lower()}_color_report.txt"
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("\n".join(report_text))

    # Confusion matrix
    cm = confusion_matrix(y_valid, y_pred, labels=labels)

    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=labels
    )
    disp.plot(
        ax=ax,
        cmap="viridis",
        values_format="d",
        colorbar=True
    )
    ax.set_title(f"Confusion Matrix - {model_name} with Color Features")
    plt.tight_layout()

    cm_path = figures_dir / f"{model_name.lower()}_color_confusion_matrix.png"
    plt.savefig(cm_path, dpi=300, bbox_inches="tight")
    plt.show()

    # Feature importance
    feature_names = get_preprocessed_feature_names(pipeline)
    importances = pipeline.named_steps["model"].feature_importances_

    feature_importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    }).sort_values(
        by="importance",
        ascending=False
    ).reset_index(drop=True)

    importance_path = tables_dir / f"{model_name.lower()}_color_feature_importance.csv"
    feature_importance_df.to_csv(
        importance_path,
        index=False,
        encoding="utf-8-sig"
    )

    # Top 20 feature importance figure
    top20 = feature_importance_df.head(20).iloc[::-1]

    plt.figure(figsize=(10, 8))
    plt.barh(top20["feature"], top20["importance"])
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.title(f"Top 20 Feature Importance - {model_name} with Color Features")
    plt.tight_layout()

    importance_fig_path = figures_dir / f"{model_name.lower()}_color_feature_importance_top20.png"
    plt.savefig(importance_fig_path, dpi=300, bbox_inches="tight")
    plt.show()

    result = {
        "Model": model_name,
        "Accuracy": accuracy,
        "Balanced Accuracy": balanced_acc,
        "Macro F1": macro_f1
    }

    return result, feature_importance_df

In [ ]:
# ============================================================
# RandomForest with color index features
# ============================================================

from sklearn.ensemble import RandomForestClassifier

preprocessor, numeric_cols, categorical_cols = build_preprocessor(X_train_color)

print("Numeric features:", len(numeric_cols))
print("Categorical features:", len(categorical_cols))
print("Color index features:", ["u_g", "g_r", "r_i", "i_z"])

rf_color_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight=None
)

rf_color_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", rf_color_model)
    ]
)

rf_color_pipeline.fit(X_train_color, y_train)

rf_color_result, rf_color_importance = evaluate_and_save_model(
    model_name="RandomForest",
    pipeline=rf_color_pipeline,
    X_valid=X_valid_color,
    y_valid=y_valid,
    output_dir=RF_COLOR_DIR,
    labels=["GALAXY", "QSO", "STAR"]
)

rf_color_result

In [ ]:
# ============================================================
# ExtraTrees with color index features
# ============================================================

from sklearn.ensemble import ExtraTreesClassifier

preprocessor, numeric_cols, categorical_cols = build_preprocessor(X_train_color)

et_color_model = ExtraTreesClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight=None
)

et_color_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", et_color_model)
    ]
)

et_color_pipeline.fit(X_train_color, y_train)

et_color_result, et_color_importance = evaluate_and_save_model(
    model_name="ExtraTrees",
    pipeline=et_color_pipeline,
    X_valid=X_valid_color,
    y_valid=y_valid,
    output_dir=ET_COLOR_DIR,
    labels=["GALAXY", "QSO", "STAR"]
)

et_color_result

In [ ]:
# ============================================================
# Color feature model comparison table
# ============================================================

color_model_comparison = pd.DataFrame([
    rf_color_result,
    et_color_result
])

color_model_comparison = color_model_comparison.sort_values(
    by="Macro F1",
    ascending=False
).reset_index(drop=True)

# Save CSV
color_model_comparison.to_csv(
    REPORT_DIR / "model_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save Markdown table without tabulate dependency
markdown_lines = []
markdown_lines.append("| Model | Accuracy | Balanced Accuracy | Macro F1 |")
markdown_lines.append("|---|---:|---:|---:|")

for _, row in color_model_comparison.iterrows():
    markdown_lines.append(
        f"| {row['Model']} | "
        f"{row['Accuracy']:.6f} | "
        f"{row['Balanced Accuracy']:.6f} | "
        f"{row['Macro F1']:.6f} |"
    )

markdown_table = "\n".join(markdown_lines)

with open(REPORT_DIR / "model_comparison.md", "w", encoding="utf-8") as f:
    f.write(markdown_table)

color_model_comparison